# Análises de Process Mining com PM4Py

Notebook de análise do event log extraído do Datajud (CNJ).  
Cobre as três categorias do **IEEE Process Mining Manifesto**:

| # | Tipo | Pergunta central |
|---|---|---|
| 1 | **Descoberta** | Qual é o fluxo real do processo? |
| 2 | **Conformidade** | O real bate com o modelo esperado? |
| 3 | **Melhoria** | Onde há gargalos, rework e desvios? |

> **Pré-requisito:** `pip install pm4py` e, para as visualizações de grafo, `brew install graphviz` (macOS).

In [ ]:
# ── Dependências ──────────────────────────────────────────────────────────────
# pip install pm4py
# macOS: brew install graphviz   (necessário para salvar DFG/Petri Net como PNG)

import glob
import os
import warnings
warnings.filterwarnings('ignore')

import pm4py
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from collections import Counter
from IPython.display import Image, display

%matplotlib inline
plt.rcParams.update({'figure.figsize': (13, 5), 'figure.dpi': 110})

# Diretório de saída das imagens geradas
os.makedirs('imgs', exist_ok=True)

print(f'PM4Py {pm4py.__version__}')

---
## 0. Carregar o Event Log

In [ ]:
# ── Encontra o arquivo XES mais recente em output/ ───────────────────────────
xes_files = sorted(glob.glob('../output/*.xes'))
if not xes_files:
    raise FileNotFoundError('Nenhum arquivo .xes encontrado. Execute main.py primeiro.')

xes_path = xes_files[-1]
print(f'Carregando: {xes_path}')


def fix_colunas(df):
    """Remove o duplo prefixo case:case: gerado pelo PM4Py ao ler XES como DataFrame.

    Em PM4Py 2.7+, read_xes() retorna um DataFrame diretamente e adiciona
    o prefixo 'case:' a todos os atributos de trace. Como nossos atributos
    já têm 'case:' (ex: case:classe), o resultado é case:case:classe.
    Esta função remove esse prefixo extra.
    """
    cols = [c[5:] if c.startswith('case:case:') else c for c in df.columns]
    return df.set_axis(cols, axis=1)


# PM4Py 2.7+ retorna DataFrame; fix_colunas precisa ser aplicado no próprio log
# para que pm4py.filter_trace_attribute_values() funcione corretamente.
log = fix_colunas(pm4py.read_xes(xes_path))
df  = log  # são o mesmo DataFrame

# ── Estatísticas gerais ───────────────────────────────────────────────────────
n_cases  = df['case:concept:name'].nunique()
n_events = len(df)
n_acts   = df['concept:name'].nunique()

print(f'\n{"="*45}')
print(f'  Processos (casos) : {n_cases:>10,}')
print(f'  Eventos           : {n_events:>10,}')
print(f'  Atividades únicas : {n_acts:>10,}')
print(f'  Eventos/processo  : {n_events/n_cases:>10.1f}')
print(f'{"="*45}')
print(f'\nColunas disponíveis:')
print([c for c in df.columns])

In [ ]:
# ── Amostra do DataFrame ──────────────────────────────────────────────────────
df.head(8)

---
## 1. Descoberta de Processo (*Process Discovery*)

**Objetivo:** construir um modelo do fluxo real a partir dos dados, sem suposições prévias.

> Sempre filtre por `case:classe` antes de descobrir o modelo.
> Misturar classes processuais diferentes gera um DFG fragmentado e ilegível.

In [ ]:
# ── Escolhe a classe com mais processos para a análise ───────────────────────
top_classes = df['case:classe'].value_counts()
print('Top 10 classes processuais:')
print(top_classes.head(10).to_string())

# Altera aqui para analisar outra classe
CLASSE_FOCO = top_classes.index[0]
print(f'\nClasse selecionada: "{CLASSE_FOCO}"')

# log já é um DataFrame fixado — filter retorna DataFrame com as mesmas colunas
log_class = pm4py.filter_trace_attribute_values(
    log, 'case:classe', [CLASSE_FOCO], retain=True
)
df_class = log_class  # mesmo DataFrame filtrado

print(f'Processos nessa classe : {df_class["case:concept:name"].nunique():,}')
print(f'Eventos nessa classe   : {len(df_class):,}')

In [ ]:
# ── DFG de Frequência ─────────────────────────────────────────────────────────
# Mostra quais atividades se sucedem e com que frequência.
# Tamanho dos nós e espessura das arestas = frequência de ocorrência.

dfg_freq, sa, ea = pm4py.discover_dfg(log_class)

# Salva como PNG (requer graphviz)
try:
    pm4py.save_vis_dfg(dfg_freq, sa, ea, 'imgs/dfg_frequencia.png')
    print('DFG salvo em imgs/dfg_frequencia.png')
    display(Image('imgs/dfg_frequencia.png'))
except Exception as e:
    print(f'Visualização indisponível (graphviz necessário): {e}')

# Mostra as 10 transições mais frequentes como alternativa
top_edges = sorted(dfg_freq.items(), key=lambda x: -x[1])[:10]
print('\nTop 10 transições:')
for (src, dst), cnt in top_edges:
    label_src = src[:35] + '…' if len(src) > 35 else src
    label_dst = dst[:35] + '…' if len(dst) > 35 else dst
    print(f'  {cnt:>6,}×  {label_src}  →  {label_dst}')

In [ ]:
# ── DFG de Performance ────────────────────────────────────────────────────────
# Arcos coloridos pelo tempo MÉDIO de espera entre atividades (em segundos).
# Arcos vermelhos/quentes = maior gargalo temporal.

dfg_perf, sa_p, ea_p = pm4py.discover_performance_dfg(log_class)

try:
    pm4py.save_vis_performance_dfg(dfg_perf, sa_p, ea_p, 'imgs/dfg_performance.png')
    print('DFG de performance salvo em imgs/dfg_performance.png')
    display(Image('imgs/dfg_performance.png'))
except Exception as e:
    print(f'Visualização indisponível: {e}')

# Top 10 transições mais lentas
perf_edges = [(k, v['mean'] / 86400) for k, v in dfg_perf.items()]
top_slow   = sorted(perf_edges, key=lambda x: -x[1])[:10]
print('\nTop 10 transições mais lentas (média em dias):')
for (src, dst), days in top_slow:
    label_src = src[:30] + '…' if len(src) > 30 else src
    label_dst = dst[:30] + '…' if len(dst) > 30 else dst
    print(f'  {days:>6.1f} dias  {label_src}  →  {label_dst}')

In [ ]:
# ── Inductive Miner → Petri Net ───────────────────────────────────────────────
# Gera um modelo formal (Petri Net) que descreve o processo.
# noise_threshold=0.2 → ignora 20% das variantes raras para simplificar o modelo.
# Aumente para modelos mais simples; reduza para capturar mais comportamentos.

net, im, fm = pm4py.discover_petri_net_inductive(log_class, noise_threshold=0.2)

try:
    pm4py.save_vis_petri_net(net, im, fm, 'imgs/petri_net.png')
    print('Petri Net salva em imgs/petri_net.png')
    display(Image('imgs/petri_net.png'))
except Exception as e:
    print(f'Visualização indisponível: {e}')

print(f'\nPlaces:      {len(net.places)}')
print(f'Transitions: {len(net.transitions)}')
print(f'Arcs:        {len(net.arcs)}')

In [ ]:
# ── BPMN (alternativa mais legível que Petri Net) ─────────────────────────────
bpmn_model = pm4py.discover_bpmn_inductive(log_class, noise_threshold=0.2)

try:
    pm4py.save_vis_bpmn(bpmn_model, 'imgs/bpmn.png')
    print('BPMN salvo em imgs/bpmn.png')
    display(Image('imgs/bpmn.png'))
except Exception as e:
    print(f'Visualização indisponível: {e}')

---
## 2. Análise de Variantes

**Variante** = sequência única de atividades seguida por um caso.  
Geralmente as **top 5 variantes cobrem 70–80%** dos casos — o restante são exceções.

Alta dispersão de variantes → falta de padronização procedimental.

In [ ]:
# ── Calcula variantes via pandas ──────────────────────────────────────────────
df_v = df_class.sort_values(['case:concept:name', 'time:timestamp'])

# Variante = sequência de atividades de cada processo
case_variant = (
    df_v.groupby('case:concept:name')['concept:name']
    .apply(lambda x: ' → '.join(x))
)
variant_counts = case_variant.value_counts()

total_cases = len(case_variant)
cum_pct     = variant_counts.cumsum() / total_cases * 100

print(f'Total de variantes : {len(variant_counts):,}')
print(f'Total de processos : {total_cases:,}')
print()

# Tabela top 15
rows = []
for rank, (variant, cnt) in enumerate(variant_counts.head(15).items(), 1):
    steps = variant.split(' → ')
    rows.append({
        'Rank': rank,
        'Casos': cnt,
        '% Total': f'{cnt/total_cases:.1%}',
        '% Acum.': f'{cum_pct[variant]:.1f}%',
        'Etapas': len(steps),
        'Início': steps[0][:40],
        'Fim':    steps[-1][:40],
    })

display(pd.DataFrame(rows).set_index('Rank'))

In [ ]:
# ── Gráfico de Pareto das Variantes ──────────────────────────────────────────
top_n = min(30, len(variant_counts))
top_v = variant_counts.head(top_n)
cumulative = (top_v.cumsum() / total_cases * 100).values

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(range(top_n), top_v.values, color='#4361ee', alpha=0.85, label='Casos')
ax2.plot(range(top_n), cumulative, color='#e63946', linewidth=2, marker='o',
         markersize=4, label='% Acumulado')
ax2.axhline(80, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax2.text(top_n * 0.98, 81, '80%', ha='right', color='gray', fontsize=9)

ax1.set_xlabel('Variante (rank)')
ax1.set_ylabel('Número de processos', color='#4361ee')
ax2.set_ylabel('% Acumulado', color='#e63946')
ax2.set_ylim(0, 105)
ax1.set_title(f'Pareto de Variantes — {CLASSE_FOCO}\nTop {top_n} de {len(variant_counts)} variantes')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.tight_layout()
plt.savefig('imgs/variantes_pareto.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantas variantes cobrem 80% dos casos?
n_80 = (cumulative < 80).sum() + 1
print(f'{n_80} variante(s) cobrem 80% dos processos.')

---
## 3. Perspectiva Temporal — Performance e Gargalos

Usa os timestamps para medir **quanto tempo** cada etapa consome.

- **Throughput time**: tempo total do processo (ajuizamento → último evento)
- **Sojourn time**: tempo de espera após cada atividade até a próxima
- **Bottleneck**: transição com maior sojourn time médio

In [ ]:
# ── Tempo de Ciclo (Throughput Time) ─────────────────────────────────────────
df_s = df_class.sort_values(['case:concept:name', 'time:timestamp'])
case_bounds = df_s.groupby('case:concept:name')['time:timestamp'].agg(['min', 'max'])
case_bounds['duration_days'] = (
    (case_bounds['max'] - case_bounds['min']).dt.total_seconds() / 86400
)
durations = case_bounds['duration_days'].dropna()

# Estatísticas
stats = durations.describe(percentiles=[.25, .5, .75, .90, .95])
print('Tempo de Ciclo (dias):')
print(f'  Média      : {stats["mean"]:>8.1f}')
print(f'  Mediana    : {stats["50%"]:>8.1f}')
print(f'  P75        : {stats["75%"]:>8.1f}')
print(f'  P90        : {stats["90%"]:>8.1f}')
print(f'  P95        : {stats["95%"]:>8.1f}')
print(f'  Máximo     : {stats["max"]:>8.1f}')

# ── Gráfico ───────────────────────────────────────────────────────────────────
cap = durations.quantile(0.95)  # corta outliers extremos para visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma
durations.clip(upper=cap).hist(bins=40, ax=axes[0], color='#4361ee',
                                edgecolor='white', linewidth=0.4)
axes[0].axvline(stats['mean'],   color='red',    ls='--', lw=1.5,
                label=f'Média: {stats["mean"]:.0f}d')
axes[0].axvline(stats['50%'],    color='orange', ls='--', lw=1.5,
                label=f'Mediana: {stats["50%"]:.0f}d')
axes[0].set_title('Distribuição do Tempo de Ciclo (truncado no P95)')
axes[0].set_xlabel('Dias')
axes[0].set_ylabel('Processos')
axes[0].legend()

# Boxplot por ano de ajuizamento
if 'case:data_ajuizamento' in df_class.columns:
    case_extra = df_class[['case:concept:name', 'case:data_ajuizamento']].drop_duplicates()
    case_extra = case_extra.merge(
        case_bounds[['duration_days']].reset_index(), on='case:concept:name'
    )
    case_extra['ano'] = pd.to_datetime(
        case_extra['case:data_ajuizamento'], errors='coerce'
    ).dt.year.astype('Int64').astype(str)
    anos_validos = case_extra.dropna(subset=['ano'])
    data_by_year = [anos_validos[anos_validos['ano'] == a]['duration_days'].values
                    for a in sorted(anos_validos['ano'].unique())]
    anos_labels  = sorted(anos_validos['ano'].unique())
    axes[1].boxplot(data_by_year, labels=anos_labels, patch_artist=True,
                    boxprops=dict(facecolor='#a8d8ea'), medianprops=dict(color='#2E4057', lw=2))
    axes[1].set_title('Duração por Ano de Ajuizamento')
    axes[1].set_xlabel('Ano')
    axes[1].set_ylabel('Dias')
    axes[1].tick_params(axis='x', rotation=45)
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.savefig('imgs/throughput_time.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sojourn Time por Atividade ────────────────────────────────────────────────
# Tempo de espera após cada atividade até o próximo movimento do mesmo processo.
# Identifica qual etapa "segura" o processo por mais tempo.

df_s = df_class.sort_values(['case:concept:name', 'time:timestamp']).copy()
df_s['next_ts'] = df_s.groupby('case:concept:name')['time:timestamp'].shift(-1)
df_s['wait_h']  = (
    (df_s['next_ts'] - df_s['time:timestamp']).dt.total_seconds() / 3600
)

sojourn = (
    df_s.dropna(subset=['wait_h'])
    .groupby('concept:name')['wait_h']
    .agg(media='mean', mediana='median', n='count')
    .assign(media_dias=lambda x: x['media'] / 24,
            mediana_dias=lambda x: x['mediana'] / 24)
    .sort_values('media_dias', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
y      = range(len(sojourn))
labels = [a[:45] + '…' if len(a) > 45 else a for a in sojourn.index]
ax.barh(y, sojourn['media_dias'],   color='#4361ee', label='Média',   alpha=0.9)
ax.barh(y, sojourn['mediana_dias'], color='#f59e0b', label='Mediana', alpha=0.75)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Dias até o próximo evento')
ax.set_title('Sojourn Time — Top 15 Atividades que mais retêm o processo')
ax.legend()
plt.tight_layout()
plt.savefig('imgs/sojourn_time.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabela resumida
display(
    sojourn[['media_dias', 'mediana_dias', 'n']]
    .rename(columns={'media_dias': 'Média (dias)', 'mediana_dias': 'Mediana (dias)', 'n': 'N'})
    .style.format({'Média (dias)': '{:.1f}', 'Mediana (dias)': '{:.1f}', 'N': '{:,}'})
    .background_gradient(subset=['Média (dias)'], cmap='YlOrRd')
)

In [ ]:
# ── Gargalos: Transições mais lentas ─────────────────────────────────────────
# Mostra pares (A → B) com maior tempo médio entre os eventos consecutivos.

df_s = df_class.sort_values(['case:concept:name', 'time:timestamp']).copy()
df_s['next_act'] = df_s.groupby('case:concept:name')['concept:name'].shift(-1)
df_s['next_ts']  = df_s.groupby('case:concept:name')['time:timestamp'].shift(-1)
df_s['wait_d']   = (
    (df_s['next_ts'] - df_s['time:timestamp']).dt.total_seconds() / 86400
)

bottlenecks = (
    df_s.dropna(subset=['next_act', 'wait_d'])
    .groupby(['concept:name', 'next_act'])['wait_d']
    .agg(media='mean', mediana='median', n='count')
    .reset_index()
    .rename(columns={'concept:name': 'De', 'next_act': 'Para'})
    .sort_values('media', ascending=False)
    .head(10)
    .assign(transicao=lambda x: x['De'].str[:30] + '  →  ' + x['Para'].str[:30])
)

fig, ax = plt.subplots(figsize=(12, 5))
y = range(len(bottlenecks))
ax.barh(y, bottlenecks['media'],   color='#e63946', label='Média',   alpha=0.9)
ax.barh(y, bottlenecks['mediana'], color='#f59e0b', label='Mediana', alpha=0.75)
ax.set_yticks(y)
ax.set_yticklabels(bottlenecks['transicao'], fontsize=9)
ax.set_xlabel('Dias')
ax.set_title('Top 10 Gargalos — Transições com maior tempo de espera')
ax.legend()
plt.tight_layout()
plt.savefig('imgs/bottlenecks.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Detecção de Rework (Loops)

**Rework** = atividade que se repete dentro do mesmo processo sem necessidade.  
Indica retrabalho, erros, litigância excessiva ou falhas procedimentais.

Exemplos comuns:
- `Conclusão ao Juiz` repetida → despachos protelatórios
- `Juntada de Citação` repetida → dificuldade de localizar a parte
- `Expedição de Mandado` repetida → tentativas frustradas de cumprimento

In [ ]:
# ── Detecção de Rework ────────────────────────────────────────────────────────
activity_per_case = (
    df_class.groupby(['case:concept:name', 'concept:name'])
    .size()
    .reset_index(name='count')
)
rework_df = activity_per_case[activity_per_case['count'] > 1]

n_affected = rework_df['case:concept:name'].nunique()
total_cls  = df_class['case:concept:name'].nunique()
print(f'Processos com rework : {n_affected:,} ({n_affected / total_cls:.1%} do total)')
print(f'Processos sem rework : {total_cls - n_affected:,} ({(total_cls - n_affected) / total_cls:.1%})')

# Top atividades com rework
top_rework = (
    rework_df.groupby('concept:name')
    .agg(total_repeticoes=('count', 'sum'),
         processos_afetados=('case:concept:name', 'nunique'),
         media_repeticoes=('count', 'mean'))
    .sort_values('total_repeticoes', ascending=False)
    .head(15)
    .round(2)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico de total de repetições
labels = [a[:40] + '…' if len(a) > 40 else a for a in top_rework.index]
axes[0].barh(labels, top_rework['total_repeticoes'], color='#e63946')
axes[0].set_xlabel('Total de repetições')
axes[0].set_title('Atividades com mais Rework\n(total de repetições)')

# Gráfico de processos afetados
axes[1].barh(labels, top_rework['processos_afetados'], color='#f59e0b')
axes[1].set_xlabel('Processos afetados')
axes[1].set_title('Atividades com mais Rework\n(processos distintos afetados)')

plt.tight_layout()
plt.savefig('imgs/rework.png', dpi=150, bbox_inches='tight')
plt.show()

display(top_rework)

---
## 5. Perspectiva Organizacional

Analisa **quem** faz cada atividade e como os recursos (varas/câmaras) se comportam.

Usa o campo `org:resource` (órgão julgador de cada movimento).

In [ ]:
# ── Volume e Performance por Vara ─────────────────────────────────────────────
if 'org:resource' not in df_class.columns:
    print('Campo org:resource não encontrado no log.')
else:
    # Volume de eventos por vara (top 20)
    vol = df_class['org:resource'].value_counts().head(20)

    # Duração mediana por vara (usa o órgão mais frequente do processo como vara principal)
    vara_principal = (
        df_class.groupby('case:concept:name')['org:resource']
        .agg(lambda x: x.value_counts().index[0])
        .rename('vara')
    )
    case_dur_vara = case_bounds[['duration_days']].join(vara_principal)
    perf_vara = (
        case_dur_vara.groupby('vara')['duration_days']
        .agg(mediana='median', n='count')
        .query('n >= 5')
        .sort_values('mediana', ascending=False)
        .head(20)
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Subplot 1: Volume de eventos
    labels_vol = [v[:45] + '…' if len(str(v)) > 45 else str(v) for v in vol.index]
    axes[0].barh(labels_vol, vol.values, color='#4361ee')
    axes[0].set_xlabel('N° de eventos')
    axes[0].set_title('Volume de Eventos por Vara (Top 20)')
    axes[0].tick_params(axis='y', labelsize=8)

    # Subplot 2: Duração mediana
    labels_dur = [v[:45] + '…' if len(str(v)) > 45 else str(v) for v in perf_vara.index]
    axes[1].barh(labels_dur, perf_vara['mediana'], color='#f59e0b')
    axes[1].set_xlabel('Duração mediana (dias)')
    axes[1].set_title('Duração Mediana por Vara\n(processos com ≥ 5 casos, top 20 mais lentas)')
    axes[1].tick_params(axis='y', labelsize=8)

    plt.tight_layout()
    plt.savefig('imgs/organizacional.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Estatísticas globais por vara
    overall = (
        case_dur_vara.groupby('vara')['duration_days']
        .agg(['median', 'mean', 'count'])
        .rename(columns={'median': 'Mediana (d)', 'mean': 'Média (d)', 'count': 'Processos'})
        .query('Processos >= 5')
        .sort_values('Mediana (d)', ascending=False)
    )
    print(f'\nVariação entre varas:')
    print(f'  Vara mais rápida (mediana): {overall["Mediana (d)"].min():.0f} dias')
    print(f'  Vara mais lenta  (mediana): {overall["Mediana (d)"].max():.0f} dias')
    print(f'  Diferença:                 {overall["Mediana (d)"].max() - overall["Mediana (d)"].min():.0f} dias')

In [ ]:
# ── Handover of Work (transferências entre órgãos) ────────────────────────────
# Conta quantas vezes o processo muda de org:resource entre eventos consecutivos.
# Alta taxa de handover pode indicar estrutura fragmentada.

if 'org:resource' in df_class.columns:
    df_h = df_class.sort_values(['case:concept:name', 'time:timestamp']).copy()
    df_h['next_resource'] = df_h.groupby('case:concept:name')['org:resource'].shift(-1)
    transfers = df_h.dropna(subset=['next_resource'])
    transfers = transfers[transfers['org:resource'] != transfers['next_resource']]

    ho_matrix = (
        transfers.groupby(['org:resource', 'next_resource'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(10)
    )

    print('Top 10 transferências entre órgãos julgadores:')
    for _, row in ho_matrix.iterrows():
        org_de  = str(row['org:resource'])[:35]
        org_para = str(row['next_resource'])[:35]
        print(f"  {row['count']:>5,}×  {org_de}  →  {org_para}")

    n_with_ho = transfers['case:concept:name'].nunique()
    print(f'\nProcessos com ao menos 1 handover: {n_with_ho:,} ({n_with_ho/total_cls:.1%})')

---
## 6. Verificação de Conformidade (*Conformance Checking*)

Compara o log real com um **modelo de referência** para detectar desvios.

Aqui usamos o próprio log para construir o modelo (Inductive Miner),  
e depois medimos o quanto cada processo se desvia dele.

**Fitness = 1.0** → processo seguiu o modelo perfeitamente.  
**Fitness < 0.8** → processo tem desvios significativos.

In [ ]:
# ── Token Replay ──────────────────────────────────────────────────────────────
# Usa a Petri Net descoberta anteriormente como modelo de referência.
# Cada token representa um "processo passando pelo modelo".
# Missing tokens = atividades que ocorreram sem passar pelo caminho esperado.

print('Aplicando token replay...')
fitness = pm4py.fitness_token_based_replay(log_class, net, im, fm)

print(f"\nFitness médio            : {fitness['average_trace_fitness']:.2%}")
print(f"Traces totalmente conformes : {fitness['percentage_of_fitting_traces']:.2%}")
print(f"Missing tokens (total)   : {fitness['missing_tokens']:,}")
print(f"Remaining tokens (total) : {fitness['remaining_tokens']:,}")

In [ ]:
# ── Diagnóstico por Processo ──────────────────────────────────────────────────
diagnostics = pm4py.conformance_diagnostics_token_based_replay(log_class, net, im, fm)

fit_rows = []
for i, (trace, diag) in enumerate(zip(log_class, diagnostics)):
    fit_rows.append({
        'processo': trace.attributes.get('concept:name', f'trace_{i}'),
        'fitness': diag['trace_fitness'],
        'missing': diag['missing_tokens'],
        'remaining': diag['remaining_tokens'],
        'n_eventos': len(trace),
    })

fit_df = pd.DataFrame(fit_rows)

# Distribuição de fitness
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(0, 1, 21)
ax.hist(fit_df['fitness'], bins=bins, color='#4361ee', edgecolor='white', lw=0.4)
ax.axvline(0.8, color='red', ls='--', lw=1.5, label='Limite 0.8')
ax.set_xlabel('Fitness')
ax.set_ylabel('N° de processos')
ax.set_title('Distribuição de Fitness — Token Replay')
ax.legend()
plt.tight_layout()
plt.savefig('imgs/conformance_fitness.png', dpi=150, bbox_inches='tight')
plt.show()

# Processos mais problemáticos
print('\n10 processos com menor fitness (mais desviantes):')
display(
    fit_df.sort_values('fitness').head(10)
    .style.format({'fitness': '{:.2%}', 'n_eventos': '{:,}'})
    .background_gradient(subset=['fitness'], cmap='RdYlGn')
)

---
## 7. Segmentação e Filtros

PM4Py oferece filtros poderosos para isolar subconjuntos do log.  
Todos os filtros retornam um novo EventLog — não modificam o original.

In [ ]:
# ── Filtros disponíveis ───────────────────────────────────────────────────────

# 1. Por atributo do trace (caso)
# log_g1 = pm4py.filter_trace_attribute_values(log, 'case:grau', ['G1'])

# 2. Por intervalo de tempo
# from datetime import datetime
# log_2024 = pm4py.filter_time_range(log,
#     datetime(2024, 1, 1), datetime(2024, 12, 31),
#     mode='traces_contained')
# Modos: 'traces_contained'  → trace inteiramente dentro do intervalo
#        'traces_intersecting' → trace tem ao menos 1 evento no intervalo

# 3. Por atividade (mantém apenas traces que passaram por essa atividade)
# log_com_audiencia = pm4py.filter_event_attribute_values(
#     log, 'concept:name', ['Audiência de Instrução'], retain=True)

# 4. Duração do trace
# log_lentos = pm4py.filter_case_performance(log,
#     min_performance=365*86400, max_performance=float('inf'))
# (performance é em segundos)

# 5. Nível de sigilo
# log_publico = pm4py.filter_trace_attribute_values(
#     log, 'case:nivel_sigilo', [0], retain=True)

# ── Exemplo: comparar duas classes processuais ────────────────────────────────
top2_classes = df['case:classe'].value_counts().head(2).index.tolist()
print(f'Comparando as 2 classes mais frequentes:')

stats_rows = []
for cls in top2_classes:
    df_c = pm4py.filter_trace_attribute_values(log, 'case:classe', [cls])
    n_c  = df_c['case:concept:name'].nunique()
    n_e  = len(df_c)

    # Duração
    bounds = df_c.sort_values('time:timestamp').groupby('case:concept:name')['time:timestamp'].agg(['min','max'])
    dur    = ((bounds['max'] - bounds['min']).dt.total_seconds() / 86400)

    # Variantes
    vcount = df_c.sort_values('time:timestamp').groupby('case:concept:name')['concept:name'].apply(tuple).value_counts()

    stats_rows.append({
        'Classe': cls[:50],
        'Processos': n_c,
        'Atividades': df_c['concept:name'].nunique(),
        'Variantes': len(vcount),
        'Eventos/Processo': round(n_e / n_c, 1),
        'Duração Mediana (d)': round(dur.median(), 1),
        'Duração P90 (d)': round(dur.quantile(0.9), 1),
    })

display(pd.DataFrame(stats_rows).set_index('Classe'))

---
## 8. Comparação entre Tribunais

Carrega os XES de todos os tribunais disponíveis em `output/`  
e compara as métricas de performance lado a lado.

In [ ]:
# ── Carrega todos os XES disponíveis ─────────────────────────────────────────
tribunal_files = {}
for f in sorted(glob.glob('../output/*.xes')):
    tribunal = os.path.basename(f).split('_')[0]
    tribunal_files[tribunal] = f  # fica com o mais recente

print(f'Tribunais encontrados: {list(tribunal_files.keys())}')

comp_rows = []
dfs_all   = {}

for tribunal, path in tribunal_files.items():
    log_t = pm4py.read_xes(path)
    df_t  = pm4py.convert_to_dataframe(log_t)
    dfs_all[tribunal] = df_t

    n_cases  = df_t['case:concept:name'].nunique()
    n_events = len(df_t)

    # Duração dos casos
    b = df_t.sort_values('time:timestamp').groupby('case:concept:name')['time:timestamp'].agg(['min','max'])
    dur = (b['max'] - b['min']).dt.total_seconds() / 86400

    # Variantes
    vcount = (
        df_t.sort_values('time:timestamp')
        .groupby('case:concept:name')['concept:name']
        .apply(tuple)
        .value_counts()
    )

    comp_rows.append({
        'Tribunal':             tribunal,
        'Processos':            n_cases,
        'Eventos':              n_events,
        'Atividades únicas':    df_t['concept:name'].nunique(),
        'Variantes':            len(vcount),
        'Eventos/Processo':     round(n_events / n_cases, 1),
        'Duração Média (d)':    round(dur.mean(), 1),
        'Duração Mediana (d)':  round(dur.median(), 1),
        'Duração P90 (d)':      round(dur.quantile(0.9), 1),
    })

comp_df = pd.DataFrame(comp_rows).set_index('Tribunal')
display(
    comp_df.style
    .format({'Processos': '{:,}', 'Eventos': '{:,}'})
    .background_gradient(subset=['Duração Mediana (d)'], cmap='YlOrRd')
)

In [ ]:
# ── Gráfico comparativo ───────────────────────────────────────────────────────
if len(comp_df) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = ['Duração Mediana (d)', 'Eventos/Processo', 'Variantes']
    titles  = ['Duração Mediana (dias)', 'Eventos por Processo', 'Variantes distintas']
    colors  = ['#4361ee', '#2dc653', '#f59e0b']

    for ax, metric, title, color in zip(axes, metrics, titles, colors):
        comp_df[metric].plot(kind='bar', ax=ax, color=color, edgecolor='white')
        ax.set_title(title)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=0)

    plt.suptitle('Comparação entre Tribunais', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('imgs/comparacao_tribunais.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Apenas 1 tribunal disponível. Execute main.py com mais tribunais em config.py.')

---
## Próximos Passos

| Análise | Função PM4Py | O que requer |
|---|---|---|
| Alignments (conformance preciso) | `pm4py.conformance_diagnostics_alignments()` | Petri Net de referência normativa |
| Decision Mining | `pm4py.discover_decision_tree()` | Modelo com pontos de decisão |
| Social Network (handover) | `pm4py.discover_handover_of_work_network()` | Campo `org:resource` populado |
| Dotted Chart | `pm4py.view_dotted_chart()` | Timestamps válidos |
| Caso vs. Modelo | `pm4py.play_out()` | Petri Net + marcação inicial |
| Enriquecimento externo | pandas merge | Tabela de metas CNJ por classe |

Para análises de **SLA compliance**, adicione `case:valor_causa` e a tabela de metas do CNJ  
(Justiça em Números) como fonte externa — veja `docs/PROCESSO.md` para detalhes.